In [9]:
import pandas as pd
import json
import time
import itertools
from functools import partial
from Dataset import train_csv_path
from models import SimpleCNN, ResNet
from validation import k_fold_validation, simple_validation
import torch
from predict import _predict_probs
from Dataset import test_csv_path, test_img_folder_path
from utils import _get_device
from models import train_model

In [2]:
df = pd.read_csv(train_csv_path)

In [3]:
simplecnn_experiments = [
    #our initial simple CNN with 3 blocks of 2 convs and dropout
    ("scnn_baseline", {"filters": (32,64,128), "n_convs": 2, "dropout": 0.4}, partial(SimpleCNN, filters=(32,64,128), n_convs=2, dropout=0.4)),
    # wider filters
    ("scnn_wide", {"filters": (64,128,256)}, partial(SimpleCNN, filters=(64,128,256), n_convs=2, dropout=0.4)),
    #3 convs per block instead of 2
    ("scnn_3conv", {"n_convs": 3}, partial(SimpleCNN, filters=(32,64,128), n_convs=3, dropout=0.4)),
    #dilated convs for wider receptive field without more params
    ("scnn_dilated", {"dilation": 2}, partial(SimpleCNN, filters=(32,64,128), n_convs=2, dropout=0.4, dilation=2)),
    #learnable downsampling with stride instead of maxpool
    ("scnn_strided", {"use_stride": True}, partial(SimpleCNN, filters=(32,64,128), n_convs=2, dropout=0.4, use_stride=True)),
]

resnet_experiments = [
    #ResNet with only 1 block per stage and no SE
    ("rn_lean", {"blocks_per_stage": 1}, partial(ResNet, filters=(32,64,128), blocks_per_stage=1, use_se=False)),
    #baseline ResNet with 2 blocks per stage and no SE
    ("rn_baseline", {"blocks_per_stage": 2, "use_se": False}, partial(ResNet, filters=(32,64,128), blocks_per_stage=2, use_se=False)),
    #wider ResNet with more filters but same number of blocks and no SE
    ("rn_wide", {"filters": (64,128,256)}, partial(ResNet, filters=(64,128,256), blocks_per_stage=2, use_se=False)),
    #ResNet with SE blocks which can improve performance with minimal extra params
    ("rn_se", {"use_se": True}, partial(ResNet, filters=(32,64,128), blocks_per_stage=2, use_se=True)),
]

configs = simplecnn_experiments + resnet_experiments

#Overview of experiments
print(f"{len(configs)} experiments\n")
print(f"{'Name':<20} {'Params':>10}  Variable tested")
print("-" * 55)
for name, changed, fn in configs:
    model = fn()
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    var = list(changed.keys())[0] if len(changed) <= 2 else "baseline"
    print(f"{name:<20} {params:>10,}  {var}")

9 experiments

Name                     Params  Variable tested
-------------------------------------------------------
scnn_baseline           288,618  baseline
scnn_wide             1,148,618  filters
scnn_3conv              482,826  n_convs
scnn_dilated            288,618  dilation
scnn_strided            482,378  use_stride
rn_lean                 308,074  blocks_per_stage
rn_baseline             696,042  blocks_per_stage
rn_wide               2,776,522  filters
rn_se                   702,154  use_se


In [4]:
# import torch
# print("CUDA available:", torch.cuda.is_available())
# print("CUDA version:", torch.version.cuda)
# print("Device count:", torch.cuda.device_count())
# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA version: 12.1
Device count: 1
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [5]:
results = []
for name, params, model_fn in configs:
    print(f"\n=== {name} ===")
    t0 = time.time()
    acc = simple_validation(df, model_fn, epochs=10, use_tta=False)
    elapsed = int(time.time() - t0)
    entry = {"name": name, "params": params, "acc": acc, "sec": elapsed}
    results.append(entry)
    print(json.dumps(entry))


=== scnn_baseline ===


Epoch 10: 100%|██████████| 107/107 [00:51<00:00,  2.09it/s]


TTA enabled: False, using 1 transforms
Validation accuracy = 99.73529577255249
{"name": "scnn_baseline", "params": {"filters": [32, 64, 128], "n_convs": 2, "dropout": 0.4}, "acc": 99.73529577255249, "sec": 490}

=== scnn_wide ===


Epoch 10: 100%|██████████| 107/107 [02:10<00:00,  1.22s/it]


TTA enabled: False, using 1 transforms
Validation accuracy = 99.58823323249817
{"name": "scnn_wide", "params": {"filters": [64, 128, 256]}, "acc": 99.58823323249817, "sec": 977}

=== scnn_3conv ===


...

TTA enabled: False, using 1 transforms
Validation accuracy = 99.58823323249817
{"name": "rn_baseline", "params": {"blocks_per_stage": 2, "use_se": false}, "acc": 99.58823323249817, "sec": 508}

=== rn_wide ===


Epoch 10: 100%|██████████| 107/107 [00:48<00:00,  2.21it/s]


TTA enabled: False, using 1 transforms
Validation accuracy = 99.32352900505066
{"name": "rn_wide", "params": {"filters": [64, 128, 256]}, "acc": 99.32352900505066, "sec": 504}

=== rn_se ===


Epoch 10: 100%|██████████| 107/107 [00:41<00:00,  2.58it/s]


TTA enabled: False, using 1 transforms
Validation accuracy = 99.647057056427
{"name": "rn_se", "params": {"use_se": true}, "acc": 99.647057056427, "sec": 451}


In [6]:
print("=== All results ===")
for r in sorted(results, key=lambda x: x["acc"], reverse=True):
    print(f"{r['name']:55s}  {r['acc']:.5f}")

=== All results ===
scnn_3conv                                               99.79412
scnn_dilated                                             99.76470
scnn_baseline                                            99.73530
rn_se                                                    99.64706
scnn_wide                                                99.58823
rn_baseline                                              99.58823
scnn_strided                                             99.55882
rn_lean                                                  99.47059
rn_wide                                                  99.32353


In [8]:
#k_fold confirmation on top 4 candidates
top4 = sorted(results, key=lambda x: x["acc"], reverse=True)[:4]
top4_names = {r["name"] for r in top4}

kfold_results = []
for name, params, model_fn in configs:
    if name not in top4_names:
        continue
    print(f"\n=== k_fold: {name} ===")
    t0 = time.time()
    fold_accs = k_fold_validation(df, model_fn, epochs=20, use_tta=False)
    elapsed = int(time.time() - t0)
    mean_acc = sum(fold_accs) / len(fold_accs)
    entry = {"name": name, "params": params, "acc": mean_acc, "folds": fold_accs, "sec": elapsed}
    kfold_results.append(entry)
    print(json.dumps(entry))

print("\n=== k_fold summary ===")
for r in sorted(kfold_results, key=lambda x: x["acc"], reverse=True):
    print(f"{r['name']:55s}  {r['acc']:.5f}")


=== k_fold: scnn_baseline ===
Fold 1/4


Epoch 20: 100%|██████████| 100/100 [00:43<00:00,  2.30it/s]


TTA enabled: False, using 1 transforms
Fold 1 accuracy = 99.85882639884949
Fold 2/4


Epoch 20: 100%|██████████| 100/100 [00:51<00:00,  1.93it/s]


TTA enabled: False, using 1 transforms
Fold 2 accuracy = 99.9294102191925
Fold 3/4


...

TTA enabled: False, using 1 transforms
Fold 2 accuracy = 99.90588426589966
Fold 3/4


Epoch 20: 100%|██████████| 100/100 [00:41<00:00,  2.40it/s]


TTA enabled: False, using 1 transforms
Fold 3 accuracy = 99.78823661804199
Fold 4/4


Epoch 20: 100%|██████████| 100/100 [00:37<00:00,  2.63it/s]


TTA enabled: False, using 1 transforms
Fold 4 accuracy = 99.74117875099182
Mean acc: 99.82941299676895 and STD 0.06732608454888651
{"name": "rn_se", "params": {"use_se": true}, "acc": 99.82941299676895, "folds": [99.88235235214233, 99.90588426589966, 99.78823661804199, 99.74117875099182], "sec": 3265}

=== k_fold summary ===
scnn_3conv                                               99.90000
scnn_baseline                                            99.84706
rn_se                                                    99.82941
scnn_dilated                                             99.81176


In [12]:
from pytorchcv.model_provider import get_model
import torch.nn as nn
def build_resnet20():
    model = get_model("resnet20_cifar10", pretrained=False)
    print(model.features.init_block)
    # adapt first conv for grayscale
    model.features.init_block.conv = nn.Conv2d(1, 16, kernel_size=3, padding=1, bias=False)
    return model

acc = simple_validation(df, build_resnet20, epochs=20, use_tta=False)
print(acc)

ConvBlock(
  (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (activ): ReLU(inplace=True)
)


Epoch 20: 100%|██████████| 107/107 [00:46<00:00,  2.32it/s]


TTA enabled: False, using 1 transforms
Validation accuracy = 99.67647194862366
99.67647194862366


In [ ]:
from utils import save_pred_to_csv

ensemble_configs = [
      ("scnn_3conv", partial(SimpleCNN, filters=(32,64,128), n_convs=3,  dropout=0.4)),
      ("scnn_baseline", partial(SimpleCNN, filters=(32,64,128), n_convs=2,  dropout=0.4)),
      ("rn_se",partial(ResNet,    filters=(32,64,128), blocks_per_stage=2, use_se=True)),
      ("scnn_dilated", partial(SimpleCNN, filters=(32,64,128), n_convs=2,  dropout=0.4, dilation=2)),
]
device = _get_device()
test_df = pd.read_csv(test_csv_path)

ensemble_probs= None
seeds = [42, 123, 456, 789, 999]
n_models=0
for name, model_fn in ensemble_configs:
    for seed in seeds:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        print(f"\nTraining {name} seed={seed}")
        model = train_model(model_fn, epochs=20, device=device)
        probs =_predict_probs(model, test_df, test_img_folder_path, test_csv_path,device=device, batch_size=128, use_tta=True)
        ensemble_probs = probs if ensemble_probs is None else ensemble_probs + probs
        n_models+=1
ensemble_probs /= n_models
predictions=ensemble_probs.argmax(dim=1).tolist()
submission = pd.DataFrame({"Id": test_df["Id"].tolist(), "Category": predictions})
path = save_pred_to_csv(submission, "ensemble_full_submission")
print(f"\nSaved to {path} ({n_models} models)")


Training scnn_3conv seed=42
